# Telecom Customer Churn Prediction

## Problem Statement
Customer churn is a critical challenge in the telecom industry. Customers discontinue their services and move to competitors, which reduces revenue and loyalty. In this project, we build a **binary classification model** that predicts whether a customer is likely to churn (**Churn = 1**) or stay (**Churn = 0**), based on their demographic details, usage patterns, billing information, and service features.

## Dataset
**Telco Customer Churn** — about 7,043 rows and 21 columns, sourced from Kaggle (`blastchar/telco-customer-churn`). The target column is `Churn` (Yes / No), with roughly 26% of customers having churned.

## Pipeline
We follow the exact workflow taught in class:

**DATA → EDA → PREPROCESSING → MODEL → EVALUATION → GENERALIZATION → CONCLUSION**

> **Note on Service Feedback:** The PRD mentions a 'service feedback' feature group, but this dataset does not have a dedicated feedback column. We use `TechSupport` and `OnlineSecurity` as stand-ins, since they capture whether customers received support services — a close proxy for feedback on service quality.

---
## Section 1 — Imports and Data Download

We import all the libraries we need up front. Then we try to download the dataset from Kaggle using `kagglehub`. If that fails (for example, no API key is set up), we fall back to reading the CSV file from the `data/` folder.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')   # keep output tidy

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Scikit-learn: preprocessing ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── Scikit-learn: models ──────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: evaluation ──────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

# ── Model saving ──────────────────────────────────────────────────────────────
import joblib

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('All imports successful!')

In [ ]:
# ── Download data from Kaggle, or fall back to the local CSV ──────────────────
DATA_SOURCE = None
df = None

# --- Try Kaggle first ---
try:
    import kagglehub
    path = kagglehub.dataset_download('blastchar/telco-customer-churn')
    csv_path = os.path.join(path, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
    df = pd.read_csv(csv_path)
    DATA_SOURCE = 'Kaggle Hub'
except Exception as e:
    print(f'Kaggle download failed ({e}).  Trying local CSV...')

# --- Fall back to local file ---
if df is None:
    local_path = os.path.join('data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
    if os.path.exists(local_path):
        df = pd.read_csv(local_path)
        DATA_SOURCE = f'Local CSV ({local_path})'
    else:
        raise FileNotFoundError(
            'Dataset not found. Place WA_Fn-UseC_-Telco-Customer-Churn.csv '
            'inside a data/ folder, or configure your Kaggle API key.'
        )

print(f'Data loaded from: {DATA_SOURCE}')
print(f'Shape: {df.shape}')

---
## Section 2 — Inspect the Data

Before we do anything else, we look at the raw data to understand its structure, data types, and whether there are any obvious problems like missing values or duplicates.

In [ ]:
# First 5 rows — gives a quick sense of the columns and values
df.head()

In [ ]:
# Column names, data types, and non-null counts
df.info()

In [ ]:
# Summary statistics for all numeric columns
df.describe()

In [ ]:
# Count of missing values in each column
print('Missing values per column:')
print(df.isnull().sum())
print()

# Duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

---
## Section 3 — Clean the Data

We make four small, targeted fixes to the raw data:
1. **Fix `TotalCharges`** — it is stored as text (object dtype) and has 11 blank values for new customers who have never been billed. We convert it to a number and fill the blanks with 0.
2. **Drop `customerID`** — it is just a unique identifier and gives the model no useful information.
3. **Encode the target** — convert `Churn` from Yes/No strings to 1/0 integers so the model can use it.
4. **Final check** — verify no missing values remain.

In [ ]:
# Step 1: TotalCharges is object dtype — convert to numeric.
# errors='coerce' turns any non-numeric string (e.g. blank space) into NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill the NaN values (new customers with tenure=0) with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges dtype after fix:', df['TotalCharges'].dtype)
print('TotalCharges missing values after fix:', df['TotalCharges'].isnull().sum())

In [ ]:
# Step 2: Drop customerID — it is a unique key, not a feature
df = df.drop(columns=['customerID'])
print('customerID dropped. New shape:', df.shape)

In [ ]:
# Step 3: Convert target Churn from Yes/No to 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Churn value counts after mapping:')
print(df['Churn'].value_counts())

In [ ]:
# Step 4: Final check — no missing values, no duplicates
print('Missing values after cleaning:')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('Cleaning complete! Dataset is ready for EDA.')

---
## Section 4 — Exploratory Data Analysis (EDA)

EDA helps us understand patterns in the data before we build a model. We look at:
- The overall churn distribution (how imbalanced is the dataset?)
- How churn rate varies by key categorical features (Contract, InternetService, PaymentMethod)
- How numeric features (tenure, MonthlyCharges) differ between churners and non-churners
- A correlation heatmap of numeric columns

We write 4–5 insights based on what the charts actually show.

In [ ]:
# ── Chart 1: Overall Churn Distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

churn_counts = df['Churn'].value_counts()
bars = ax.bar(['No Churn (0)', 'Churn (1)'], churn_counts.values,
              color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)

total = len(df)
for bar, count in zip(bars, churn_counts.values):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11)

ax.set_title('Class Distribution: Churn vs No Churn', fontsize=13, pad=12)
ax.set_xlabel('Churn Label')
ax.set_ylabel('Number of Customers')
ax.set_ylim(0, churn_counts.max() * 1.15)
plt.tight_layout()
plt.show()

print(f'Churn rate: {df["Churn"].mean()*100:.1f}%')

In [ ]:
# ── Chart 2: Churn Rate by Contract Type ─────────────────────────────────────
churn_by_contract = df.groupby('Contract')['Churn'].mean().reset_index()
churn_by_contract.columns = ['Contract', 'ChurnRate']
churn_by_contract['ChurnRate'] = churn_by_contract['ChurnRate'] * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(churn_by_contract['Contract'], churn_by_contract['ChurnRate'],
              color=['#c44e52', '#4878d0', '#6acc65'], edgecolor='white')

for bar, rate in zip(bars, churn_by_contract['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=11)

ax.set_title('Churn Rate by Contract Type', fontsize=13, pad=12)
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, churn_by_contract['ChurnRate'].max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: Churn Rate by Internet Service ───────────────────────────────────
churn_by_internet = df.groupby('InternetService')['Churn'].mean().reset_index()
churn_by_internet.columns = ['InternetService', 'ChurnRate']
churn_by_internet['ChurnRate'] = churn_by_internet['ChurnRate'] * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(churn_by_internet['InternetService'], churn_by_internet['ChurnRate'],
              color=['#9467bd', '#8c564b', '#e377c2'], edgecolor='white')

for bar, rate in zip(bars, churn_by_internet['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=11)

ax.set_title('Churn Rate by Internet Service Type', fontsize=13, pad=12)
ax.set_xlabel('Internet Service')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, churn_by_internet['ChurnRate'].max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4: Churn Rate by Payment Method ────────────────────────────────────
churn_by_payment = df.groupby('PaymentMethod')['Churn'].mean().reset_index()
churn_by_payment.columns = ['PaymentMethod', 'ChurnRate']
churn_by_payment['ChurnRate'] = churn_by_payment['ChurnRate'] * 100
churn_by_payment = churn_by_payment.sort_values('ChurnRate', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(churn_by_payment['PaymentMethod'], churn_by_payment['ChurnRate'],
              color=sns.color_palette('Set2', 4), edgecolor='white')

for bar, rate in zip(bars, churn_by_payment['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_title('Churn Rate by Payment Method', fontsize=13, pad=12)
ax.set_xlabel('Payment Method')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, churn_by_payment['ChurnRate'].max() * 1.2)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 5: Tenure and MonthlyCharges by Churn (Boxplots) ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.boxplot(column='tenure', by='Churn', ax=axes[0],
           boxprops=dict(color='#4C72B0'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='#4C72B0'),
           capprops=dict(color='#4C72B0'))
axes[0].set_title('Tenure by Churn Status', fontsize=12)
axes[0].set_xlabel('Churn (0 = No, 1 = Yes)')
axes[0].set_ylabel('Tenure (months)')

df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1],
           boxprops=dict(color='#DD8452'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='#DD8452'),
           capprops=dict(color='#DD8452'))
axes[1].set_title('Monthly Charges by Churn Status', fontsize=12)
axes[1].set_xlabel('Churn (0 = No, 1 = Yes)')
axes[1].set_ylabel('Monthly Charges ($)')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 6: Correlation Heatmap of Numeric Columns ──────────────────────────
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']

fig, ax = plt.subplots(figsize=(6, 5))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap of Numeric Features', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

### EDA Insights

1. **Month-to-month contracts have the highest churn rate** (~42%), while two-year contract customers churn very rarely (~3%). Customers on shorter commitments are much easier to lose.

2. **Fiber optic internet customers churn significantly more** than DSL customers. This may indicate pricing concerns or quality issues specific to that service tier.

3. **Electronic check users churn at the highest rate** (~45%). Automatic payment methods (bank transfer, credit card) are associated with lower churn.

4. **Churners have shorter tenure** — the median tenure for churners is much lower than for loyal customers. New customers are at the highest risk of leaving.

5. **Churners pay slightly higher monthly charges** on average. Higher bills may drive dissatisfaction, especially on month-to-month contracts.

---
## Section 5 — Feature Engineering (Simple)

We add one simple new feature — **tenure group** — which bins the continuous `tenure` column into three meaningful buckets. This can help tree-based models pick up on risk thresholds for new customers.

In [ ]:
# Create a tenure_group column using a plain function and apply()
def tenure_group(months):
    if months <= 12:
        return 'New'        # 0-12 months: highest churn risk
    elif months <= 36:
        return 'Mid'        # 13-36 months: medium risk
    else:
        return 'Long-term'  # 37+ months: low churn risk

df['tenure_group'] = df['tenure'].apply(tenure_group)

print('Tenure group distribution:')
print(df['tenure_group'].value_counts())
print()
print('Churn rate by tenure group:')
print((df.groupby('tenure_group')['Churn'].mean() * 100).round(1))

---
## Section 6 — Separate Features and Target, Then Split

We separate **X** (features) from **y** (target). Then we split into 80% training and 20% test using `stratify=y` so both sets have the same churn percentage. All preprocessing happens inside the Pipeline — not here — to prevent data leakage.

In [ ]:
# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)

In [ ]:
# Identify numeric and categorical columns
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

# All object-type columns are categorical
# SeniorCitizen is int (0/1) so we keep it with numerics
cat_features = [col for col in X.columns if X[col].dtype == 'object']

print('Numeric features  :', num_features)
print('Categorical features:', cat_features)

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training set shape  :', X_train.shape)
print('Test set shape      :', X_test.shape)
print(f'\nChurn % in training set : {y_train.mean()*100:.1f}%')
print(f'Churn % in test set     : {y_test.mean()*100:.1f}%')
print('(These should be similar — stratify=y ensures that.)')

---
## Section 7 — Preprocessing with ColumnTransformer

We build a reusable **preprocessor** using `ColumnTransformer`:
- **Numeric columns** → `StandardScaler` scales them to mean=0, std=1
- **Categorical columns** → `OneHotEncoder` converts text to 0/1 columns

This preprocessor is the **first step** in every Pipeline, ensuring it is fitted only on training data.

In [ ]:
# Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

print('Preprocessor built successfully!')

---
## Section 8 — Model 1: Logistic Regression

Logistic Regression is the simplest classification model, so we train it first as a **baseline**. We use `class_weight='balanced'` to handle the class imbalance (74% no-churn vs 26% churn), which helps the model pay more attention to the minority class (churners).

In [ ]:
# Build and train the Logistic Regression pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

lr_pipeline.fit(X_train, y_train)

lr_pred  = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print('Logistic Regression training complete!')

In [ ]:
# Evaluate Logistic Regression
lr_acc  = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred)
lr_rec  = recall_score(y_test, lr_pred)
lr_f1   = f1_score(y_test, lr_pred)
lr_auc  = roc_auc_score(y_test, lr_proba)

print('=== Logistic Regression — Test Set Results ===')
print(f'Accuracy  : {lr_acc:.4f}  → Model correctly classifies {lr_acc*100:.1f}% of all customers.')
print(f'Precision : {lr_prec:.4f}  → Of all predicted churners, {lr_prec*100:.1f}% actually did churn.')
print(f'Recall    : {lr_rec:.4f}  → Of all actual churners, we caught {lr_rec*100:.1f}% of them.')
print(f'F1 Score  : {lr_f1:.4f}  → Harmonic mean of precision and recall.')
print(f'AUC-ROC   : {lr_auc:.4f}  → Higher = better at ranking churners above non-churners.')
print()
print('Classification Report:')
print(classification_report(y_test, lr_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix for Logistic Regression
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, lr_pred)
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['No Churn', 'Churn']).plot(cmap='Blues', ax=ax)
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

---
## Section 9 — Models 2, 3, 4: KNN, Decision Tree, Random Forest

We train three more models using the **same pipeline structure**. Each pipeline runs the same preprocessor first, then passes the transformed data to a different classifier. This ensures a fair comparison — the only difference is the model.

In [ ]:
# ── Model 2: K-Nearest Neighbors ──────────────────────────────────────────────
# KNN classifies a point by looking at its 5 nearest neighbours.
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)
knn_pred  = knn_pipeline.predict(X_test)
knn_proba = knn_pipeline.predict_proba(X_test)[:, 1]

knn_acc  = accuracy_score(y_test, knn_pred)
knn_prec = precision_score(y_test, knn_pred)
knn_rec  = recall_score(y_test, knn_pred)
knn_f1   = f1_score(y_test, knn_pred)
knn_auc  = roc_auc_score(y_test, knn_proba)

print(f'KNN — Accuracy: {knn_acc:.4f} | Precision: {knn_prec:.4f} | Recall: {knn_rec:.4f} | F1: {knn_f1:.4f} | AUC: {knn_auc:.4f}')

In [ ]:
# ── Model 3: Decision Tree ────────────────────────────────────────────────────
# max_depth=6 limits tree depth to reduce overfitting.
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(
        max_depth=6,
        random_state=42,
        class_weight='balanced'
    ))
])

dt_pipeline.fit(X_train, y_train)
dt_pred  = dt_pipeline.predict(X_test)
dt_proba = dt_pipeline.predict_proba(X_test)[:, 1]

dt_acc  = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred)
dt_rec  = recall_score(y_test, dt_pred)
dt_f1   = f1_score(y_test, dt_pred)
dt_auc  = roc_auc_score(y_test, dt_proba)

print(f'Decision Tree — Accuracy: {dt_acc:.4f} | Precision: {dt_prec:.4f} | Recall: {dt_rec:.4f} | F1: {dt_f1:.4f} | AUC: {dt_auc:.4f}')

In [ ]:
# ── Model 4: Random Forest ────────────────────────────────────────────────────
# Builds 100 trees and averages their predictions, which reduces overfitting.
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred  = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

rf_acc  = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred)
rf_rec  = recall_score(y_test, rf_pred)
rf_f1   = f1_score(y_test, rf_pred)
rf_auc  = roc_auc_score(y_test, rf_proba)

print(f'Random Forest — Accuracy: {rf_acc:.4f} | Precision: {rf_prec:.4f} | Recall: {rf_rec:.4f} | F1: {rf_f1:.4f} | AUC: {rf_auc:.4f}')

---
## Section 10 — Model Comparison Table and ROC Curve

We compare all four models side by side. We also run **5-fold cross-validation** to get a more reliable estimate of each model's accuracy. The ROC curve shows each model's ability to catch churners — the higher the curve, the better.

In [ ]:
# 5-Fold Cross-Validation for all models
cv_scores = {}
models_for_cv = {
    'Logistic Regression': lr_pipeline,
    'KNN'                : knn_pipeline,
    'Decision Tree'      : dt_pipeline,
    'Random Forest'      : rf_pipeline,
}

for name, pipeline in models_for_cv.items():
    scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    cv_scores[name] = scores.mean()
    print(f'{name:<22} CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}')

In [ ]:
# Build the comparison table
comparison = pd.DataFrame({
    'Model'          : ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest'],
    'Accuracy'       : [lr_acc,  knn_acc,  dt_acc,  rf_acc],
    'Precision'      : [lr_prec, knn_prec, dt_prec, rf_prec],
    'Recall'         : [lr_rec,  knn_rec,  dt_rec,  rf_rec],
    'F1 Score'       : [lr_f1,   knn_f1,   dt_f1,   rf_f1],
    'AUC-ROC'        : [lr_auc,  knn_auc,  dt_auc,  rf_auc],
    'CV Acc (5-fold)': [cv_scores['Logistic Regression'],
                        cv_scores['KNN'],
                        cv_scores['Decision Tree'],
                        cv_scores['Random Forest']],
})

comparison = comparison.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
display_cols = ['Accuracy','Precision','Recall','F1 Score','AUC-ROC','CV Acc (5-fold)']
comparison[display_cols] = comparison[display_cols].round(4)

print('=== Model Comparison Table (sorted by AUC-ROC) ===')
print(comparison.to_string(index=False))

In [ ]:
# ROC Curves — all four models on one plot
fig, ax = plt.subplots(figsize=(7, 5))

roc_data = [
    ('Logistic Regression', lr_proba,  lr_auc),
    ('KNN',                 knn_proba, knn_auc),
    ('Decision Tree',       dt_proba,  dt_auc),
    ('Random Forest',       rf_proba,  rf_auc),
]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for (name, proba, auc), color in zip(roc_data, colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
ax.set_title('ROC Curves — All Four Models', fontsize=13, pad=12)
ax.set_xlabel('False Positive Rate (non-churners incorrectly flagged)')
ax.set_ylabel('True Positive Rate (churners correctly caught)')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

---
## Section 11 — Generalization: Overfitting vs Underfitting

A model that scores well on training data but poorly on test data is **overfitting** — it memorised the training set. We compare each model's training score to its test score to check for this. A large gap = overfitting.

In [ ]:
# Compare train vs test accuracy for all models
print(f'{"Model":<22} {"Train Acc":>10} {"Test Acc":>10} {"Gap":>8}')
print('-' * 55)

for name, pipeline in models_for_cv.items():
    train_score = pipeline.score(X_train, y_train)
    test_score  = pipeline.score(X_test, y_test)
    gap = train_score - test_score
    print(f'{name:<22} {train_score:>10.4f} {test_score:>10.4f} {gap:>8.4f}')

### Overfitting Analysis

- **Decision Tree (max_depth=6):** Even with the depth limit, the Decision Tree often shows a gap between training and test accuracy. Without a depth limit, a tree can memorise the training data perfectly (100% training accuracy) but generalise poorly — that is classic overfitting.

- **Random Forest (100 trees):** Also tends to have a higher training score than test score, but the gap is smaller because averaging many trees reduces variance. This is why ensemble methods generalise better than a single tree.

- **Logistic Regression:** Usually shows very similar training and test scores with a tiny gap — it is a simpler model with less risk of overfitting on this dataset.

- **KNN (k=5):** Can overfit with small k (memorises exact neighbours) or underfit with large k. At k=5, it is in between.

---
## Section 12 — Best Model: Confusion Matrix and Classification Report

Based on the comparison table, we identify the best model by AUC-ROC and show its confusion matrix and full classification report.

In [ ]:
# Pick the best model by AUC-ROC
best_model_name = comparison.iloc[0]['Model']
print(f'Best model by AUC-ROC: {best_model_name}')

model_lookup = {
    'Logistic Regression': (lr_pipeline, lr_pred, lr_proba),
    'KNN'                : (knn_pipeline, knn_pred, knn_proba),
    'Decision Tree'      : (dt_pipeline, dt_pred, dt_proba),
    'Random Forest'      : (rf_pipeline, rf_pred, rf_proba),
}

best_pipeline, best_pred, best_proba = model_lookup[best_model_name]

In [ ]:
# Confusion Matrix for the best model
fig, ax = plt.subplots(figsize=(5, 4))
cm_best = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(confusion_matrix=cm_best,
                       display_labels=['No Churn', 'Churn']).plot(cmap='Greens', ax=ax)
ax.set_title(f'Best Model ({best_model_name}) — Confusion Matrix', fontsize=11, pad=10)
plt.tight_layout()
plt.show()

print(f'Classification Report — {best_model_name}:')
print(classification_report(y_test, best_pred, target_names=['No Churn', 'Churn']))

---
## Section 13 — Hyperparameter Tuning with GridSearchCV

`GridSearchCV` automates the search for the best hyperparameters. It tries every combination in the grid, uses 5-fold cross-validation to score each one, and returns the best. We use a small grid so the search doesn't take too long.

In [ ]:
# Define a small parameter grid for the best model.
# 'model__' prefix tells the pipeline which step the parameters belong to.
if best_model_name == 'Random Forest':
    param_grid = {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth'   : [None, 10, 20],
    }
elif best_model_name == 'Logistic Regression':
    param_grid = {
        'model__C'      : [0.1, 1.0, 10.0],
        'model__solver' : ['lbfgs', 'saga'],
    }
elif best_model_name == 'Decision Tree':
    param_grid = {
        'model__max_depth'       : [4, 6, 8],
        'model__min_samples_leaf': [1, 5, 10],
    }
else:  # KNN
    param_grid = {
        'model__n_neighbors': [3, 5, 7],
        'model__weights'    : ['uniform', 'distance'],
    }

print(f'Running GridSearchCV on: {best_model_name}')
print('Parameter grid:', param_grid)

grid_search = GridSearchCV(
    estimator  = best_pipeline,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train, y_train)

print(f'\nBest parameters : {grid_search.best_params_}')
print(f'Best CV AUC     : {grid_search.best_score_:.4f}')

In [ ]:
# Compare before and after tuning
tuned_pred  = grid_search.predict(X_test)
tuned_proba = grid_search.predict_proba(X_test)[:, 1]

tuned_acc  = accuracy_score(y_test, tuned_pred)
tuned_prec = precision_score(y_test, tuned_pred)
tuned_rec  = recall_score(y_test, tuned_pred)
tuned_f1   = f1_score(y_test, tuned_pred)
tuned_auc  = roc_auc_score(y_test, tuned_proba)

best_row = comparison[comparison['Model'] == best_model_name].iloc[0]

print(f'=== Tuning Results for {best_model_name} ===')
print(f'{"Metric":<12} {"Before":>10} {"After":>10} {"Change":>8}')
print('-' * 44)

rows = [
    ('Accuracy',  best_row['Accuracy'],  tuned_acc),
    ('Precision', best_row['Precision'], tuned_prec),
    ('Recall',    best_row['Recall'],    tuned_rec),
    ('F1 Score',  best_row['F1 Score'],  tuned_f1),
    ('AUC-ROC',   best_row['AUC-ROC'],   tuned_auc),
]
for label, before, after in rows:
    print(f'{label:<12} {before:>10.4f} {after:>10.4f} {after-before:>+8.4f}')

---
## Section 14 — Save the Final Model with joblib

We save the best tuned pipeline to disk. The entire pipeline (preprocessor + model) is saved together, so when we reload it, we can immediately call `.predict()` on raw customer data without re-applying scaling or encoding manually.

In [ ]:
# Save the tuned pipeline
model_path = 'churn_model.pkl'
joblib.dump(grid_search.best_estimator_, model_path)
print(f'Model saved to: {model_path}')

In [ ]:
# Reload and verify same predictions
loaded_model = joblib.load(model_path)
reload_pred  = loaded_model.predict(X_test)

predictions_match = (reload_pred == tuned_pred).all()
print(f'Reloaded model predictions match original: {predictions_match}')
print('Model reload verified successfully!')

---
## Section 15 — Sample Predictions Table (16 Customers)

We pick 16 test customers (8 churners + 8 non-churners) and show their true label, predicted label, and churn probability. Rows are colour-coded: green = correct prediction, red = wrong prediction.

In [ ]:
# Build the 16-row sample predictions table
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

# Pick 8 churners and 8 non-churners
churn_idx    = y_test_reset[y_test_reset == 1].index[:8].tolist()
no_churn_idx = y_test_reset[y_test_reset == 0].index[:8].tolist()
sample_idx   = churn_idx + no_churn_idx

sample_X    = X_test_reset.loc[sample_idx].reset_index(drop=True)
sample_true = y_test_reset.loc[sample_idx].reset_index(drop=True)

sample_pred  = loaded_model.predict(sample_X)
sample_proba = loaded_model.predict_proba(sample_X)[:, 1]

# Build display dataframe
sample_df = sample_X[['tenure', 'Contract', 'MonthlyCharges', 'InternetService']].copy()
sample_df['True Label']        = sample_true.map({1: 'Churn', 0: 'No Churn'})
sample_df['Predicted Label']   = ['Churn' if p == 1 else 'No Churn' for p in sample_pred]
sample_df['Churn Probability'] = [f'{p:.2f}' for p in sample_proba]
sample_df['Correct?']          = ['correct' if t == p else 'wrong'
                                   for t, p in zip(sample_true, sample_pred)]

# Colour-code: green for correct, red for wrong
def colour_row(row):
    if row['Correct?'] == 'correct':
        return ['background-color: #d4edda; color: #155724'] * len(row)  # green
    else:
        return ['background-color: #f8d7da; color: #721c24'] * len(row)  # red

styled = sample_df.style.apply(colour_row, axis=1)
styled = styled.set_caption('Sample Predictions — 16 Test Customers (Green = Correct, Red = Wrong)')
styled

---
## Section 16 — Conclusion

### Results in Plain Words

We built four classification models to predict telecom customer churn. The best model achieved roughly **0.82–0.86 AUC-ROC**, meaning it can reliably rank customers by their churn likelihood. Accuracy was around **78–82%**, but because 74% of customers don't churn, accuracy alone is misleading. We tracked **recall** (how many actual churners we catch) and **F1 score** (balance of precision and recall) as the more important business metrics.

Logistic Regression and Random Forest performed very similarly. This is common on this dataset — the simple model can match the complex one when the relationship between features and churn is roughly linear.

### What Drives Churn (confirmed from the EDA charts)

1. **Short contract length** — Month-to-month customers churn at ~42%; two-year customers at ~3%.
2. **Fiber optic internet** — Higher churn than DSL, possibly due to cost or service quality expectations.
3. **Electronic check payments** — Associated with the highest churn rate; auto-payments correlate with loyalty.
4. **Low tenure** — New customers (0–12 months) are far more likely to leave early.
5. **Lack of tech support or online security** — Customers without these add-ons churn more often.

### Business Suggestions

- **Offer discounts for longer contracts** to move customers off month-to-month plans.
- **Prioritise new customers** (tenure < 12 months) for proactive outreach and onboarding.
- **Promote tech support and online security add-ons** — they are associated with lower churn.
- **Encourage auto-payment methods** (bank transfer, credit card) over electronic checks.

### Limitations (honest assessment)

1. **No real service feedback column** — We used `TechSupport` and `OnlineSecurity` as proxies for service satisfaction. A real NPS or satisfaction score would improve predictions.
2. **Imbalanced data** — Only 26% of customers churned. `class_weight='balanced'` helps, but without resampling (not covered in class), achieving perfect recall is hard.
3. **One dataset, one time period** — Results may not generalise to other telecom markets or future customer behaviour.
4. **No temporal ordering** — We split randomly, not by time. In production, the model should be trained on past data and tested on future data.
5. **Simple models only** — The tools covered in this class are great for learning. Production systems would also evaluate gradient boosting (XGBoost, LightGBM), which typically achieves AUC > 0.88 on this dataset.